# Execução da camada Gold — ENEM por UF

Este notebook executa a construção da camada **Gold** para as edições de **1998 a 2025**, utilizando os arquivos Silver da Unidade Federativa definida em `UF_EXECUCAO`.

### Regras de operação
- **Parametrização:** para processar outra UF, altere apenas `UF_EXECUCAO` na Seção 1.
- **Execução centralizada:** toda a construção da Gold é realizada por `executar_gold()`, implementada em `enem_pipeline.gold`.
- **Referências territoriais:** a rotina Gold define as perspectivas aplicáveis a cada ano e base de origem.
- **Integridade:** a rotina verifica o schema e a contagem de registros antes de concluir cada saída.
- **Manifesto:** os arquivos Gold gerados são registrados no manifesto da UF.
- **Fail-fast:** a execução é interrompida caso falte uma fonte Silver ou ocorra uma falha no processamento.

> **Nota:** este notebook apenas configura, executa e resume o processo. Não replica as regras nem as funções já implementadas em `gold.py`. As camadas Bronze, Staging e Silver não são modificadas.


## 1. Configuração e Importação de Módulos

Carrega as configurações do projeto, importa a rotina de execução Gold, normaliza a UF selecionada e define os diretórios de origem e destino.

In [1]:
from _setup import *

from enem_pipeline.config import (
    PROJECT_ROOT,
    ANO_INICIAL,
    ANO_FINAL,
    SILVER_DIR,
    GOLD_DIR,
    METADATA_DIR,
    normalizar_uf,
)

from enem_pipeline.gold import executar_gold

from pathlib import Path
import pandas as pd

UF_EXECUCAO = "RR"
UF_EXECUCAO = normalizar_uf(UF_EXECUCAO)

DIRETORIO_SILVER_UF = SILVER_DIR / f"uf={UF_EXECUCAO}"
DIRETORIO_GOLD_UF = GOLD_DIR / f"uf={UF_EXECUCAO}"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTO_GOLD = METADATA_DIR / f"manifesto_gold_{UF_EXECUCAO}.csv"

print("Raiz do projeto:", PROJECT_ROOT)
print("UF selecionada:", UF_EXECUCAO)
print("Período:", f"{ANO_INICIAL}–{ANO_FINAL}")
print("Origem Silver:", DIRETORIO_SILVER_UF)
print("Destino Gold:", DIRETORIO_GOLD_UF)
print("Manifesto:", MANIFESTO_GOLD)


Raiz do projeto: /home/akel/PycharmProjects/Projeto_ENEM
UF selecionada: RR
Período: 1998–2025
Origem Silver: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR
Destino Gold: /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR
Manifesto: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/manifesto_gold_RR.csv


## 2. Conferência dos Parâmetros de Execução

Confere o período de processamento, a existência do módulo `gold.py` e a disponibilidade das 30 fontes Silver esperadas para a série histórica (26 arquivos de 1998–2023 e quatro bases de 2024–2025).

In [2]:
assert ANO_INICIAL == 1998
assert ANO_FINAL == 2025
assert len(UF_EXECUCAO) == 2
assert (PROJECT_ROOT / "src" / "enem_pipeline" / "gold.py").exists()

ANOS = list(range(ANO_INICIAL, ANO_FINAL + 1))

fontes_silver = []

for ano in ANOS:
    diretorio_ano = DIRETORIO_SILVER_UF / f"ano={ano}"

    bases = (
        [("enem", f"enem_{ano}.parquet")]
        if ano <= 2023
        else [
            ("participantes", "participantes.parquet"),
            ("resultados", "resultados.parquet"),
        ]
    )

    for base, nome_arquivo in bases:
        fontes_silver.append({
            "ano": ano,
            "base": base,
            "arquivo": diretorio_ano / nome_arquivo,
        })

fontes_ausentes = [
    fonte for fonte in fontes_silver
    if not fonte["arquivo"].is_file()
]

print(f"UF: {UF_EXECUCAO}")
print(f"Anos a percorrer: {len(ANOS)}")
print(f"Fontes Silver esperadas: {len(fontes_silver)}")
print(f"Fontes Silver encontradas: {len(fontes_silver) - len(fontes_ausentes)}")

if fontes_ausentes:
    display(pd.DataFrame([
        {
            "ano": fonte["ano"],
            "base": fonte["base"],
            "arquivo_ausente": str(fonte["arquivo"]),
        }
        for fonte in fontes_ausentes
    ]))
    raise FileNotFoundError("Existem fontes Silver ausentes. A execução Gold não será iniciada.")

assert len(fontes_silver) == 30
print("\n✓ Parâmetros e fontes Silver conferidos.")


UF: RR
Anos a percorrer: 28
Fontes Silver esperadas: 30
Fontes Silver encontradas: 30

✓ Parâmetros e fontes Silver conferidos.


## 3. Processamento Silver → Gold

Executa `executar_gold()` para toda a série histórica da UF selecionada. A rotina `gold.py` identifica as referências territoriais aplicáveis, materializa os arquivos Parquet, valida schema e contagem de registros e grava o manifesto Gold.

In [3]:
try:
    MANIFESTO_GERADO = executar_gold(
        uf=UF_EXECUCAO,
        ano_ini=ANO_INICIAL,
        ano_fim=ANO_FINAL,
        metadata=METADATA_DIR,
    )

except Exception as erro:
    print(
        f"FALHA na execução Gold: "
        f"{type(erro).__name__}: {erro}"
    )
    print("Execução interrompida. Corrija a falha antes de continuar.")
    raise

print("\nExecução Gold concluída.")
print("Manifesto gerado:", MANIFESTO_GERADO)


[16:29:42] Gold [1/48] 1998 | enem | RESIDENCIA
[16:29:42] Gold [2/48] 1999 | enem | RESIDENCIA
[16:29:42] Gold [3/48] 2000 | enem | RESIDENCIA
[16:29:42] Gold [4/48] 2001 | enem | ESC
[16:29:42] Gold [5/48] 2002 | enem | ESC
[16:29:42] Gold [6/48] 2003 | enem | ESC
[16:29:43] Gold [7/48] 2004 | enem | ESC
[16:29:43] Gold [8/48] 2005 | enem | ESC
[16:29:43] Gold [9/48] 2006 | enem | ESC
[16:29:43] Gold [10/48] 2007 | enem | ESC
[16:29:43] Gold [11/48] 2008 | enem | ESC
[16:29:43] Gold [12/48] 2008 | enem | PROVA
[16:29:43] Gold [13/48] 2009 | enem | ESC
[16:29:43] Gold [14/48] 2009 | enem | PROVA
[16:29:43] Gold [15/48] 2010 | enem | ESC
[16:29:43] Gold [16/48] 2010 | enem | PROVA
[16:29:43] Gold [17/48] 2011 | enem | ESC
[16:29:43] Gold [18/48] 2011 | enem | PROVA
[16:29:43] Gold [19/48] 2012 | enem | ESC
[16:29:43] Gold [20/48] 2012 | enem | PROVA
[16:29:43] Gold [21/48] 2013 | enem | ESC
[16:29:43] Gold [22/48] 2013 | enem | PROVA
[16:29:43] Gold [23/48] 2014 | enem | ESC
[16:29:43]

## 4. Resumo e Auditoria da Execução

Lê o manifesto produzido pela rotina Gold e apresenta um resumo das saídas registradas, referências territoriais, bases de origem e existência dos arquivos gerados.

Para o período completo, são esperadas **48 saídas Gold**: uma referência territorial por fonte entre 1998 e 2007; duas entre 2008 e 2023; e seis saídas para 2024–2025, considerando as diferenças entre `participantes` e `resultados`.

In [4]:
manifesto_gold = pd.read_csv(
    MANIFESTO_GERADO,
    sep=";",
    encoding="utf-8",
)

if manifesto_gold.empty:
    raise RuntimeError("O manifesto Gold foi gerado sem registros.")

manifesto_gold["gold_existe"] = manifesto_gold["arquivo_gold"].apply(
    lambda caminho: Path(caminho).is_file()
)

manifesto_gold["silver_existe"] = manifesto_gold["arquivo"].apply(
    lambda caminho: Path(caminho).is_file()
)

colunas_resumo = [
    "ano",
    "base",
    "tipo_referencia_territorial",
    "arquivo_gold",
    "n_registros",
    "gold_existe",
    "silver_existe",
]

display(manifesto_gold[colunas_resumo])

print("\nResumo da execução Gold")
print("Saídas registradas:", len(manifesto_gold))
print("Edições processadas:", manifesto_gold["ano"].nunique())
print(
    "Referências territoriais:",
    ", ".join(sorted(manifesto_gold["tipo_referencia_territorial"].dropna().unique()))
)
print("Bases de origem:", ", ".join(sorted(manifesto_gold["base"].dropna().unique())))
print("Arquivos Gold existentes:", int(manifesto_gold["gold_existe"].sum()))
print("Fontes Silver existentes:", int(manifesto_gold["silver_existe"].sum()))


,ano,base,tipo_referencia_territorial,arquivo_gold,n_registros,gold_existe,silver_existe
0,1998,enem,RESIDENCIA,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,967,True,True
1,1999,enem,RESIDENCIA,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,1014,True,True
2,2000,enem,RESIDENCIA,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,317,True,True
3,2001,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,2606,True,True
4,2002,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,5497,True,True
5,2003,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,6316,True,True
6,2004,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,4793,True,True
7,2005,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,5986,True,True
8,2006,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,2810,True,True
9,2007,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,4207,True,True



Resumo da execução Gold
Saídas registradas: 48
Edições processadas: 28
Referências territoriais: ESC, PROVA, RESIDENCIA
Bases de origem: enem, participantes, resultados
Arquivos Gold existentes: 48
Fontes Silver existentes: 48


## 5. Critérios de Conclusão

A execução desta etapa é considerada bem-sucedida quando:

1. O processamento de todos os anos termina sem lançar exceções;
2. O manifesto Gold é gerado para a UF selecionada;
3. O manifesto registra exatamente as **48 saídas esperadas** para o período de 1998 a 2025;
4. Todos os arquivos Gold registrados existem no destino;
5. Todas as fontes Silver referenciadas no manifesto existem.

A rotina `executar_gold()` também verifica, antes de registrar cada saída, a conformidade do schema canônico e a igualdade da contagem de registros entre Silver e Gold.

Os resultados ficam armazenados em `data/gold/uf={UF_EXECUCAO}/`. Para processar outra UF, altere `UF_EXECUCAO` na Seção 1 e reexecute o notebook.

In [5]:
assert MANIFESTO_GERADO.is_file(), "Manifesto Gold não encontrado."
assert len(manifesto_gold) == 48, (
    f"Quantidade inesperada de saídas Gold: {len(manifesto_gold)} (esperado: 48)."
)
assert manifesto_gold["arquivo_gold"].is_unique, (
    "Existem caminhos de saída Gold duplicados no manifesto."
)
assert manifesto_gold["gold_existe"].all(), (
    "Um ou mais arquivos Gold registrados não foram encontrados."
)
assert manifesto_gold["silver_existe"].all(), (
    "Uma ou mais fontes Silver registradas não foram encontradas."
)

print("=" * 64)
print("CRITÉRIOS DE CONCLUSÃO — GOLD")
print("=" * 64)
print(f"✓ UF processada: {UF_EXECUCAO}")
print(f"✓ Edições: {manifesto_gold['ano'].nunique()}")
print(f"✓ Saídas Gold registradas: {len(manifesto_gold)}/48")
print(f"✓ Arquivos Gold existentes: {int(manifesto_gold['gold_existe'].sum())}/48")
print(f"✓ Fontes Silver existentes: {int(manifesto_gold['silver_existe'].sum())}/48")
print(f"✓ Manifesto: {MANIFESTO_GERADO}")
print("\n✓ EXECUÇÃO GOLD CONCLUÍDA COM SUCESSO.")


CRITÉRIOS DE CONCLUSÃO — GOLD
✓ UF processada: RR
✓ Edições: 28
✓ Saídas Gold registradas: 48/48
✓ Arquivos Gold existentes: 48/48
✓ Fontes Silver existentes: 48/48
✓ Manifesto: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/manifesto_gold_RR.csv

✓ EXECUÇÃO GOLD CONCLUÍDA COM SUCESSO.
